In [3]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Лабораторна робота №2. Частина 2: Статистичний аналіз та побудова регресійних моделей VHI
**Виконав:** Студент групи ФБ-46 — Ільченко Влад

### Автоматичне завантаження, розпакування датасету та ініціалізація бази даних Pandas

In [4]:
import os
import zipfile
import requests
import pandas as pd
import numpy as np
from scipy import stats

def download_and_prepare_data_v2():
    folder_path = "vhi_data"
    zip_name = "vhi_data.zip"
    
    # Автозавантаження архіву з GitHub, якщо папки немає
    if not os.path.exists(folder_path):
        print("Локальну папку з даними не знайдено. Запускаємо автоматичне завантаження...")
        url = "https://raw.githubusercontent.com/vladilchen-ipt28-source/zpad_2026_labs/main/vhi_data.zip"
        
        try:
            response = requests.get(url, stream=True)
            with open(zip_name, "wb") as f:
                f.write(response.content)
            
            print(" Розпакування архіву у фоновому режимі...")
            with zipfile.ZipFile(zip_name, "r") as zip_ref:
                zip_ref.extractall(folder_path)
                
            os.remove(zip_name)  # Видаляємо тимчасовий zip, щоб репозиторій був чистим
            print("Дані успішно завантажено та підготовлено!")
        except Exception as e:
            print(f" Помилка автозавантаження: {e}")
            
    # Зчитування конвеєром Pandas
    all_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    combined_data = []
    
    for i, file_name in enumerate(all_files):
        file_path = os.path.join(folder_path, file_name)
        temp_df = pd.read_csv(file_path, skiprows=1, sep=',', on_bad_lines='skip', index_col=False)
        temp_df.columns = [c.replace(' ', '').replace(',', '').strip() for c in temp_df.columns]
        
        rename_dict = {}
        for col in temp_df.columns:
            if 'year' in col.lower(): rename_dict[col] = 'Year'
            elif 'week' in col.lower(): rename_dict[col] = 'Week'
            elif 'vhi' in col.lower() or '%' in col.lower(): rename_dict[col] = 'VHI'
        temp_df.rename(columns=rename_dict, inplace=True)
        
        temp_df.dropna(subset=['Year', 'Week', 'VHI'], inplace=True)
        temp_df = temp_df[pd.to_numeric(temp_df['Year'], errors='coerce').notnull()]
        
        temp_df['Year'] = temp_df['Year'].astype(int)
        temp_df['Week'] = temp_df['Week'].astype(int)
        temp_df['VHI'] = temp_df['VHI'].astype(float)
        temp_df['Area_ID'] = i + 1
        combined_data.append(temp_df)
        
    return pd.concat(combined_data, ignore_index=True)

# Створюємо глобальний DataFrame для другої частини
df_v2 = download_and_prepare_data_v2()
print(f"\nБаза даних готова до статистичного аналізу!")
print(f"Загальна кількість доступних рядків: {len(df_v2)}")


База даних готова до статистичного аналізу!
Загальна кількість доступних рядків: 60345


### Завдання 1. Визначити відсоток площі області, що постраждала від сильних посух (VHI < 20) за вказаний рік.

In [5]:
def calculate_drought_percentage(dataframe, area_id, year, vhi_threshold=20.0):
    """Обчислює відсоток тижнів у році, коли область страждала від сильної посухи"""
    # Фільтруємо дані за конкретну область та рік
    year_data = dataframe[(dataframe['Area_ID'] == area_id) & (dataframe['Year'] == year)]
    
    if year_data.empty:
        return 0.0
        
    # Рахуємо скільки разів індекс падав нижче норми відносно загальної кількості спостережень
    drought_weeks = year_data[year_data['VHI'] < vhi_threshold]
    percentage = (len(drought_weeks) / len(year_data)) * 100
    return round(percentage, 2)

In [6]:
# Тестуємо розрахунок для Області №9 (Київська) за посушливий 2007 рік
area = 9
year = 2007
pct = calculate_drought_percentage(df_v2, area, year)

print(f"РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 1:")
print(f"У {year} році Область ID {area} перебувала в стані сильної посухи {pct}% часу досліджуваного періоду.")

РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 1:
У 2007 році Область ID 9 перебувала в стані сильної посухи 0.0% часу досліджуваного періоду.


### Завдання 2. Побудувати лінійну регресію залежності індексу VHI від часу (років) для визначення довгострокових кліматичних трендів.

In [7]:
def build_vhi_trend(dataframe, area_id):
    """
    Використовує SciPy для розрахунку лінійної регресії тренду зміни VHI.
    Повертає коефіцієнт нахилу (slope), коефіцієнт детермінації (r_value) та p-value.
    """
    # Групуємо дані по роках, щоб отримати середньорічний VHI для стабільного тренду
    area_data = dataframe[dataframe['Area_ID'] == area_id]
    yearly_avg = area_data.groupby('Year')['VHI'].mean().reset_index()
    
    x = yearly_avg['Year']
    y = yearly_avg['VHI']
    
    # Виклик математичного апарату SciPy
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    
    return slope, r_value**2, p_value

In [8]:
# Аналізуємо кліматичний тренд для Області ID 9
slope, r_squared, p_val = build_vhi_trend(df_v2, area_id=9)

print(f"РЕЗУЛЬТАТ СТАТИСТИЧНОГО АНАЛІЗУ SciPy (ЗАВДАННЯ 2):")
print(f"Коефіцієнт нахилу тренду (Slope): {slope:.4f}")
print(f"Коефіцієнт детермінації (R-squared): {r_squared:.4f}")
print(f"Статистична значущість (p-value): {p_val:.4f}")

if p_val < 0.05:
    trend_type = "зростання індексу (покращення)" if slope > 0 else "деградації (клімат стає посушливішим)"
    print(f"-> Висновок: Знайдено статистично значущий тренд {trend_type}.")
else:
    print("-> Висновок: Чіткого довгострокового лінійного тренду не виявлено (зміни в межах статистичної похибки).")

РЕЗУЛЬТАТ СТАТИСТИЧНОГО АНАЛІЗУ SciPy (ЗАВДАННЯ 2):
Коефіцієнт нахилу тренду (Slope): 0.0765
Коефіцієнт детермінації (R-squared): 0.0305
Статистична значущість (p-value): 0.2630
-> Висновок: Чіткого довгострокового лінійного тренду не виявлено (зміни в межах статистичної похибки).
